# Milestone 4 — Model Evaluation, Comparison & Insights

This final notebook:
1. Evaluates all trained models on the **held-out test set**
2. Produces confusion matrices for each model
3. Plots ROC curves for model comparison
4. Generates a feature importance analysis (RF + XGBoost)
5. Builds a summary comparison table
6. Draws conclusions and retention strategy recommendations

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              roc_curve, classification_report)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

MODELS  = '../models'
FIGURES = '../reports/figures'
os.makedirs(FIGURES, exist_ok=True)

## 1. Load Test Data & Models

In [ ]:
def load_test(prefix, path='../data/processed'):
    X = pd.read_csv(f'{path}/{prefix}_X_test.csv')
    y = pd.read_csv(f'{path}/{prefix}_y_test.csv').values.ravel()
    return X, y

X_test_t, y_test_t = load_test('telco')
X_test_i, y_test_i = load_test('india')
print(f'Telco test : {X_test_t.shape} | India test : {X_test_i.shape}')

model_names = ['logistic_regression', 'decision_tree', 'random_forest', 'svm', 'xgboost']
display_names = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'SVM', 'XGBoost']

def load_models(prefix):
    models = {}
    for mn, dn in zip(model_names, display_names):
        path = f'{MODELS}/{prefix}_{mn}.pkl'
        if os.path.exists(path):
            with open(path, 'rb') as f:
                models[dn] = pickle.load(f)
    return models

telco_models = load_models('telco')
india_models = load_models('india')
print('Loaded models:', list(telco_models.keys()))

## 2. Test Set Evaluation

In [ ]:
def test_all(models, X_test, y_test, dataset_label):
    rows = []
    for name, model in models.items():
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        rows.append({
            'Model'     : name,
            'Dataset'   : dataset_label,
            'Accuracy'  : round(accuracy_score(y_test, y_pred), 4),
            'Precision' : round(precision_score(y_test, y_pred, zero_division=0), 4),
            'Recall'    : round(recall_score(y_test, y_pred, zero_division=0), 4),
            'F1-Score'  : round(f1_score(y_test, y_pred, zero_division=0), 4),
            'ROC-AUC'   : round(roc_auc_score(y_test, y_prob), 4)
        })
    return pd.DataFrame(rows).sort_values('F1-Score', ascending=False).reset_index(drop=True)

telco_test_df = test_all(telco_models, X_test_t, y_test_t, 'Telco')
india_test_df = test_all(india_models, X_test_i, y_test_i, 'India')

print('=== TELCO — Test Set Results ===')
display(telco_test_df)
print('\n=== INDIA — Test Set Results ===')
display(india_test_df)

## 3. Confusion Matrices

In [ ]:
def plot_confusion_matrices(models, X_test, y_test, dataset_label, filename):
    n = len(models)
    fig, axes = plt.subplots(1, n, figsize=(n * 3.5, 4))
    if n == 1:
        axes = [axes]
    
    for ax, (name, model) in zip(axes, models.items()):
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['No Churn', 'Churn'],
                    yticklabels=['No Churn', 'Churn'],
                    linewidths=0.5, cbar=False)
        ax.set_title(name, fontweight='bold', fontsize=9)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    
    plt.suptitle(f'{dataset_label} — Confusion Matrices (Test Set)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/{filename}', bbox_inches='tight')
    plt.show()

plot_confusion_matrices(telco_models, X_test_t, y_test_t, 'Telco', '12_telco_confusion.png')
plot_confusion_matrices(india_models, X_test_i, y_test_i, 'India', '13_india_confusion.png')

## 4. ROC Curves

In [ ]:
def plot_roc_curves(models, X_test, y_test, dataset_label, filename):
    fig, ax = plt.subplots(figsize=(7, 6))
    colors = sns.color_palette('tab10', len(models))
    
    for (name, model), color in zip(models.items(), colors):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
    
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate (Recall)')
    ax.set_title(f'{dataset_label} — ROC Curves (Test Set)', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.02])
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/{filename}', bbox_inches='tight')
    plt.show()

plot_roc_curves(telco_models, X_test_t, y_test_t, 'Telco', '14_telco_roc.png')
plot_roc_curves(india_models, X_test_i, y_test_i, 'India', '15_india_roc.png')

## 5. Feature Importance Analysis

In [ ]:
def plot_feature_importance(model, feature_names, model_name, dataset_label, filename, top_n=15):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])
    else:
        print(f'  {model_name}: no feature importance available')
        return
    
    feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    feat_df = feat_df.nlargest(top_n, 'Importance')
    
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = sns.color_palette('Blues_d', top_n)
    ax.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1], color=colors)
    ax.set_title(f'{dataset_label} — {model_name}: Top {top_n} Features', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/{filename}', bbox_inches='tight')
    plt.show()
    return feat_df

with open(f'{MODELS}/telco_feature_names.pkl', 'rb') as f:
    telco_features = pickle.load(f)
with open(f'{MODELS}/india_feature_names.pkl', 'rb') as f:
    india_features = pickle.load(f)

print('=== Random Forest Feature Importance — Telco ===')
rf_telco_fi = plot_feature_importance(
    telco_models['Random Forest'], telco_features,
    'Random Forest', 'Telco', '16_telco_rf_importance.png')

print('=== XGBoost Feature Importance — Telco ===')
xgb_telco_fi = plot_feature_importance(
    telco_models['XGBoost'], telco_features,
    'XGBoost', 'Telco', '17_telco_xgb_importance.png')

In [ ]:
print('=== Random Forest Feature Importance — India ===')
rf_india_fi = plot_feature_importance(
    india_models['Random Forest'], india_features,
    'Random Forest', 'India', '18_india_rf_importance.png')

print('=== XGBoost Feature Importance — India ===')
xgb_india_fi = plot_feature_importance(
    india_models['XGBoost'], india_features,
    'XGBoost', 'India', '19_india_xgb_importance.png')

## 6. Summary Comparison Table

In [ ]:
all_test = pd.concat([telco_test_df, india_test_df], ignore_index=True)
print('=== FULL MODEL COMPARISON TABLE (Test Set) ===')
display(all_test[['Dataset', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']])

all_test.to_csv(f'{MODELS}/test_results.csv', index=False)

In [ ]:
# Visual comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, (df, title) in zip(axes, [(telco_test_df, 'Telco Dataset'), (india_test_df, 'India Dataset')]):
    x = np.arange(len(df))
    width = 0.15
    colors = sns.color_palette('tab10', len(metrics))
    
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax.bar(x + i * width, df[metric], width, label=metric, color=color, alpha=0.85)
    
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(df['Model'], rotation=20, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.12)
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Model Performance Comparison — Test Set', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES}/20_model_comparison.png', bbox_inches='tight')
plt.show()

## 7. Conclusions & Retention Strategy Insights

### Model Performance — Test Set Results

#### Telco Dataset (IBM-inspired, US)
| Rank | Model | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|---|---|
| 1 | **Logistic Regression** | 0.7531 | 0.5220 | **0.8036** | **0.6329** | 0.8535 |
| 2 | **XGBoost** | 0.7796 | 0.5693 | 0.6893 | 0.6236 | **0.8559** |
| 3 | Decision Tree | 0.7682 | 0.5499 | 0.6893 | 0.6117 | 0.8256 |
| 4 | SVM | 0.7711 | 0.5565 | 0.6679 | 0.6071 | 0.8159 |
| 5 | Random Forest | **0.7852** | **0.5930** | 0.6036 | 0.5982 | 0.8330 |

> **Key insight (Telco):** Logistic Regression leads on F1 and Recall — it catches 80% of actual churners, making it the best early-warning model. XGBoost leads on ROC-AUC (0.856), making it the best ranking/scoring model. Both complement each other.

#### India Dataset (Indian Market)
| Rank | Model | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|---|---|---|---|---|---|---|
| 1 | **XGBoost** | **0.6951** | 0.6901 | 0.6863 | **0.6882** | **0.7695** |
| 2 | **Random Forest** | 0.6911 | 0.6835 | 0.6890 | 0.6862 | 0.7642 |
| 3 | Logistic Regression | 0.6720 | 0.6605 | **0.6809** | 0.6705 | 0.7380 |
| 4 | SVM | 0.6707 | 0.6622 | 0.6700 | 0.6661 | 0.7335 |
| 5 | Decision Tree | 0.6813 | **0.6900** | 0.6355 | 0.6616 | 0.7373 |

> **Key insight (India):** XGBoost is the clear winner — best F1 (0.688), best AUC (0.770), best accuracy. Random Forest is nearly identical, confirming ensemble methods' superiority for this domain.

### Key Churn Drivers Identified

**Telco Dataset (from RF/XGBoost feature importance):**
- **Contract Type (Month-to-Month)** — top predictor; ~42% churn rate vs <10% for longer contracts
- **Tenure Months** — short-tenure customers (0–12 months) at extreme risk
- **Monthly Charges** — higher charges strongly correlated with churn  
- **Internet Service (Fiber Optic)** — ~42% churn rate vs 19% DSL
- **Payment Method (Electronic Check)** — highest churn among payment types

**India Dataset (from RF/XGBoost feature importance):**
- **Overall Satisfaction** — strongest single predictor; low scores → high churn  
- **Price Sensitivity Score** — high-sensitivity customers churn proactively
- **Competitor Offers Received** — 3+ offers received dramatically increases risk  
- **Support Calls Last Month** — high call frequency signals service dissatisfaction
- **Tenure Months** — new customers (< 12 months) most vulnerable

### Recommended Data-Driven Retention Strategies

1. **Deploy Early Warning System** — Run XGBoost monthly to score all customers; flag top 20% risk for proactive outreach
2. **Target New Customers (Tenure < 12m)** — Assign dedicated relationship managers for the first year; check-in calls at months 3, 6, 9
3. **Contract Migration Campaigns** — Offer month-to-month customers a discounted 1-year contract; feature importance confirms this is the highest-leverage intervention
4. **Satisfaction Recovery Program** — Customers with overall_satisfaction ≤ 4 (India) or high monthly charges + fiber (Telco) get priority resolution SLAs
5. **Competitor Intelligence Response** — When 2+ competitor offers detected, trigger a retention offer within 48 hours
6. **Payment Method Nudge (Telco)** — Incentivize electronic-check users to switch to auto-pay (e.g. 5% monthly discount)

In [ ]:
print('Milestone 4 — Evaluation COMPLETE')
print(f'All figures saved to: {FIGURES}')
for f in sorted(os.listdir(FIGURES)):
    print(' ', f)